# 11. 승격 스코어보드

## 연구 질문

> 승격한 production 파이프라인이 **랩에서 잰 점수를 그대로 재현하고**, 리더보드에 낼
> 제출물을 만들 수 있는가?

09번은 피처셋 config가 재현성 계약을 지킨다는 것을, 10번은 조립 순서와 fit 경계가
train/test 대칭을 만든다는 것을 닫았다. 둘 다 **구조가 옳다**는 진술이다.
그러나 구조가 옳다고 점수가 같다는 보장은 없다. 컬럼 하나의 순서, 통계 하나의 자유도,
imputer를 fit한 행 범위 — 어느 하나만 어긋나도 RandomForest는 다른 트리를 만든다.

이 노트북은 그 마지막 간격을 닫는다. 노트북 08이 기록한 2024 holdout 점수를
production 코드로 다시 계산하고(G3), 통과하면 첫 제출 후보를 만든다.

**G3가 실패하면 제출하지 않는다.** 재현 실패는 조립 계약이 깨졌다는 뜻이고,
그 상태의 제출물은 무엇을 측정하는지 알 수 없다.

## 목차

1. 재현이란 무엇을 어디까지 맞추는 것인가 (Decision Box ⑪)
2. 2024 holdout 실험 설계와 경계 행 1건 (Decision Box ⑫)
3. 점수 체인 — 랩 사다리가 그대로 재현되는가 (G3)
4. 사다리를 다시 읽는다 — 개선분은 어디서 왔는가
5. 제출 후보 S1·S2 만들기 (Decision Box ⑬⑭⑮)
6. 종합 결론

## 이 노트북의 전제

- 공식 원자료는 Git에 커밋하지 않으므로 로컬 `data/raw/open/`에서 읽는다.
- 09·10과 달리 이 노트북은 **산출물을 만든다.** 모델·metadata·registry 행·제출 CSV가
  `outputs/` 아래에 생성되며, 이 경로는 `.gitignore`로 커밋에서 제외된다.
- **제출 버튼은 사람이 직접 누른다.** 이 노트북은 검증을 통과한 CSV와 ledger 행까지만 만든다.
- 설계 근거는 `docs/design/05-feature-set-promotion.md` v1.0의 7.2·8절을 따른다.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def resolveProjectRoot():
  """worktree/일반 체크아웃 어디서 실행하든 src와 원자료를 찾는다."""
  here = Path.cwd().resolve()
  for candidate in [here, *here.parents]:
    if (candidate / "src" / "baram").is_dir():
      return candidate
  raise RuntimeError("src/baram을 찾지 못했습니다")


def resolveOfficialDataDir(projectRoot):
  for candidate in [projectRoot, *projectRoot.parents]:
    dataDir = candidate / "data" / "raw" / "open"
    if (dataDir / "train" / "train_labels.csv").is_file():
      return dataDir
  raise RuntimeError("공식 데이터 디렉터리를 찾지 못했습니다")


projectRoot = resolveProjectRoot()
if str(projectRoot / "src") not in sys.path:
  sys.path.insert(0, str(projectRoot / "src"))
dataDir = resolveOfficialDataDir(projectRoot)
outputRoot = projectRoot / "outputs"

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

print(f"project root : {projectRoot.name}")
print(f"official data: {dataDir}")
print(f"outputs      : {outputRoot} (gitignore 대상)")

project root : codex-to-claude-transition-a9327d
official data: C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\data\raw\open
outputs      : C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs (gitignore 대상)


09·10과 같은 탐색 규약이다. 여기에 `outputs/` 경로가 하나 추가됐다.
이 노트북부터는 파일을 만들기 때문이다.

---

## 1. 재현이란 무엇을 어디까지 맞추는 것인가

설계서 7.2절은 G3의 허용 오차를 `1e-9`로 정했다. 그런데 무엇과 무엇의 차이를
`1e-9` 이내로 맞춰야 하는지는 생각보다 미묘하다.

In [2]:
# 설계서 1.1절 표에 적힌 값(6자리 반올림)과, 노트북 08 출력에 남은 전체 정밀도 값
LAB_REPORTED = {
  "lab_official_mean": 0.576938,
  "expanded_vector": 0.586482,
  "spatial_idw2_all_group": 0.592485,
  "spatial_nearest_own_group": 0.589828,
}

# 노트북 08 셀 26 출력: primary 후보의 전체 정밀도와 두 개의 델타
IDW_P2_ALL_FULL = 0.5924854297224146
DELTA_VS_VECTOR = 0.006002949852530692
LEAN_DELTA_VS_VECTOR = 0.0033458901465431845

LAB_FULL_PRECISION = {
  "expanded_vector": IDW_P2_ALL_FULL - DELTA_VS_VECTOR,
  "spatial_idw2_all_group": IDW_P2_ALL_FULL,
  "spatial_nearest_own_group": IDW_P2_ALL_FULL - DELTA_VS_VECTOR + LEAN_DELTA_VS_VECTOR,
}

precisionFrame = pd.DataFrame(
  [
    (
      name,
      LAB_REPORTED[name],
      LAB_FULL_PRECISION.get(name),
      None if name not in LAB_FULL_PRECISION else LAB_FULL_PRECISION[name] - LAB_REPORTED[name],
    )
    for name in LAB_REPORTED
  ],
  columns=["feature_set", "설계서 표기(6자리)", "노트북 08 전체 정밀도", "표기 오차"],
)
with pd.option_context("display.float_format", lambda value: f"{value:.16g}"):
  display(precisionFrame)

,feature_set,설계서 표기(6자리),노트북 08 전체 정밀도,표기 오차
0,lab_official_mean,0.576938,NaN,NaN
1,expanded_vector,0.5864819999999999,0.5864824798698839,4.798698839403315e-07
2,spatial_idw2_all_group,0.592485,0.5924854297224146,4.297224145410894e-07
3,spatial_nearest_own_group,0.589828,0.5898283700164271,3.700164270536987e-07


### Decision Box ⑪ — 재현 판정은 문서의 6자리 표기가 아니라 노트북 08의 전체 정밀도 값에 건다

**선택지**

- (A) 설계서 1.1절 표의 `0.592485`와 비교하고 `1e-9` 기준을 적용한다
- (B) 노트북 08 출력에 남은 `0.5924854297224146`과 비교하고 `1e-9` 기준을 적용한다
- (C) 6자리로 반올림해서 같으면 통과로 본다

**근거**

(A)는 실패한다. 그러나 그 실패는 재현이 깨졌기 때문이 아니라 **비교 대상이 이미
반올림된 값**이기 때문이다. 위 표에서 보듯 표기 오차만으로 최대 5e-7가 발생하며,
이는 `1e-9` 기준을 자동으로 넘긴다. 이 상태로 G3를 돌리면 정상 재현을 실패로 오판하고,
멀쩡한 조립 계약을 뜯어보게 된다.

(C)는 반대 방향으로 위험하다. 6자리만 맞으면 통과이므로 1e-7 규모의 실제 계산 차이
— 예를 들어 `std_ddof`가 0에서 1로 바뀐 것 같은 진짜 계약 위반 — 를 놓칠 수 있다.

(B)는 랩이 실제로 계산한 부동소수점 값을 기준으로 삼는다. 노트북 08은 채택 후보의
전체 정밀도와 두 개의 델타를 출력으로 남겨 두었으므로, 나머지 두 프리셋의 기준값도
역산할 수 있다.

**채택: (B)** — 전체 정밀도 기준 `1e-9`. 다만 `lab_official_mean`은 노트북 08이
전체 정밀도를 남기지 않았으므로 6자리 대조만 가능하며, 이 한계를 결론에 명시한다.

---

## 2. 2024 holdout 실험 설계와 경계 행 1건

재현하려면 노트북 08과 **완전히 같은 실험 조건**을 만들어야 한다.
그 조건 중 하나가 10번 노트북에서 발견한 라벨 시각 규약이다.

In [3]:
from baram.baseline import OFFICIAL_RF_PARAMS
from baram.feature_config import FeatureSetConfig, get_feature_set
from baram.feature_pipeline import build_feature_pipeline
from baram.features.turbine_metadata import load_turbine_locations
from baram.features.weather_grid import SPATIAL_STATISTICS
from baram.metrics import CAPACITY_KWH, TARGET_COLS, metric


def readOfficial(relativePath):
  return pd.read_csv(dataDir / relativePath, encoding="utf-8-sig")


trainLabels = readOfficial("train/train_labels.csv")
trainLabels["kst_dtm"] = pd.to_datetime(trainLabels["kst_dtm"])
ldapsTrain = readOfficial("train/ldaps_train.csv")
gfsTrain = readOfficial("train/gfs_train.csv")
# validator가 원본 문자열과 대조하므로 sample_submission은 읽은 그대로 둔다.
sampleSubmission = readOfficial("sample_submission.csv")
submissionTimeIndex = pd.to_datetime(sampleSubmission["forecast_kst_dtm"])
turbines = load_turbine_locations(dataDir / "info.xlsx")

labelYear = trainLabels["kst_dtm"].dt.year
yearProfile = (
  labelYear.value_counts()
  .sort_index()
  .rename_axis("연도")
  .rename("라벨 행 수")
  .to_frame()
)
yearProfile["역할"] = ["학습", "학습", "2024 holdout", "경계 행 — 어디에도 쓰지 않음"]
print(f"라벨 시각 범위 : {trainLabels['kst_dtm'].min()} ~ {trainLabels['kst_dtm'].max()}")
print(f"제출 시각 범위 : {submissionTimeIndex.min()} ~ {submissionTimeIndex.max()}")
yearProfile

라벨 시각 범위 : 2022-01-01 01:00:00 ~ 2025-01-01 00:00:00
제출 시각 범위 : 2025-01-01 01:00:00 ~ 2026-01-01 00:00:00


,라벨 행 수,역할
연도,,
2022,8759,학습
2023,8760,학습
2024,8784,2024 holdout
2025,1,경계 행 — 어디에도 쓰지 않음


### Decision Box ⑫ — 2025-01-01 00:00 경계 행 1건은 학습에도 holdout에도 넣지 않는다

**선택지**

- (A) 2024 holdout을 `year >= 2024`로 잡아 경계 행을 holdout에 포함한다
- (B) `year == 2024`로 잡아 경계 행을 어느 쪽에도 넣지 않는다
- (C) 경계 행을 2024 학습 쪽에 붙인다

**근거**

공식 라벨은 01:00에서 시작해 각 날의 00:00을 전날 마지막 시각으로 셈한다. 그래서
2025-01-01 00:00 한 행이 `train_labels`에 남고, 연도 분포가 8,759 / 8,760 / 8,784 / 1이 된다.

(A)는 holdout을 8,785행으로 만들어 노트북 08의 8,784행과 어긋난다. 단 한 행이지만
NMAE는 평균이므로 분모가 바뀌고, 그 결과 재현은 즉시 실패한다.
(C)는 holdout 기간 직후의 값을 학습에 넣는 것이라 시간 순서를 거스른다.

(B)는 노트북 08의 `modelTrainMask = year < 2024`, `holdoutMask = year == 2024`와 정확히
같다. 경계 행은 holdout 실험에서만 빠지며, **최종 제출 모델을 학습할 때는 정상적으로
포함된다** — 그때는 2025년 전체가 예측 대상이고 이 행은 과거이기 때문이다.

**채택: (B)**

제출 시각 범위도 함께 확인해 둔다. 학습 라벨의 마지막(2025-01-01 00:00)과
제출 시간축의 시작이 겹치지 않아야 look-ahead 없이 전체 학습을 쓸 수 있다.

In [4]:
HOLDOUT_YEAR = 2024

modelTrainMask = labelYear < HOLDOUT_YEAR
holdoutMask = labelYear == HOLDOUT_YEAR
boundaryMask = labelYear > HOLDOUT_YEAR


def yearSlice(frame, year, comparison):
  forecast = pd.to_datetime(frame["forecast_kst_dtm"], errors="raise")
  keep = forecast.dt.year < year if comparison == "lt" else forecast.dt.year == year
  return frame[keep]


splitFrames = {
  "train_ldaps": yearSlice(ldapsTrain, HOLDOUT_YEAR, "lt"),
  "train_gfs": yearSlice(gfsTrain, HOLDOUT_YEAR, "lt"),
  "holdout_ldaps": yearSlice(ldapsTrain, HOLDOUT_YEAR, "eq"),
  "holdout_gfs": yearSlice(gfsTrain, HOLDOUT_YEAR, "eq"),
}

holdoutSplitBounds = {
  "train_start": trainLabels.loc[modelTrainMask, "kst_dtm"].min(),
  "train_end": trainLabels.loc[modelTrainMask, "kst_dtm"].max(),
  "holdout_start": trainLabels.loc[holdoutMask, "kst_dtm"].min(),
  "holdout_end": trainLabels.loc[holdoutMask, "kst_dtm"].max(),
}
lastTrainLabel = trainLabels["kst_dtm"].max()
firstSubmission = submissionTimeIndex.min()

print(f"학습 행(2022~2023) : {int(modelTrainMask.sum())}  (노트북 08: 17519)")
print(f"holdout 행(2024)   : {int(holdoutMask.sum())}  (노트북 08: 8784)")
print(f"경계 행(2025)      : {int(boundaryMask.sum())}  -> 어느 쪽에도 넣지 않음")
print(f"합계 대조          : {int(modelTrainMask.sum() + holdoutMask.sum() + boundaryMask.sum())} == {len(trainLabels)}")
print()
print(f"학습 라벨 마지막 시각 : {lastTrainLabel}")
print(f"제출 시간축 첫 시각   : {firstSubmission}")
print(f"두 구간이 겹치지 않는가 : {lastTrainLabel < firstSubmission}")
pd.DataFrame(
  [(name, *frame.shape) for name, frame in splitFrames.items()],
  columns=["분할", "rows", "cols"],
)

학습 행(2022~2023) : 17519  (노트북 08: 17519)
holdout 행(2024)   : 8784  (노트북 08: 8784)
경계 행(2025)      : 1  -> 어느 쪽에도 넣지 않음
합계 대조          : 26304 == 26304

학습 라벨 마지막 시각 : 2025-01-01 00:00:00
제출 시간축 첫 시각   : 2025-01-01 01:00:00
두 구간이 겹치지 않는가 : True


,분할,rows,cols
0,train_ldaps,280304,35
1,train_gfs,157671,40
2,holdout_ldaps,140544,35
3,holdout_gfs,79056,40


분할이 노트북 08과 정확히 일치한다. 학습 17,519행, holdout 8,784행, 경계 행 1건 제외다.
학습 라벨의 마지막 시각(2025-01-01 00:00)이 제출 시간축의 첫 시각(2025-01-01 01:00)보다
앞서므로, 최종 모델에 라벨 전체를 써도 미래를 보는 일은 없다.

---

## 3. 점수 체인 — 랩 사다리가 그대로 재현되는가

프리셋 4종을 같은 조건에서 학습하고 2024 holdout 점수를 잰다.
여기에 두 개를 더한다 — 랩 통제군 74개는 프리셋이 아니므로 ad-hoc config로 만들고,
production 기본값인 `official_mean`(76개) 프리셋도 따로 잰다. 둘은 lead feature 2개만큼
다르며, S1 제출물이 쓰는 것은 **후자**이므로 ledger에 정직한 숫자를 남기려면 둘 다 필요하다.

In [5]:
# 랩 통제군(74개)은 lead가 없다. official_mean 프리셋(76개)과는 다른 정의이므로
# Decision Box ①의 결정에 따라 여기서만 ad-hoc config로 재현한다.
labControl = FeatureSetConfig(name="lab_official_mean", statistics=("mean",), include_lead=False)

scoreConfigs = {
  "lab_official_mean": labControl,
  # production 기본값(76개). 랩 통제군과 lead 2개만큼 다르므로 점수도 따로 잰다.
  "official_mean": get_feature_set("official_mean"),
  "expanded_vector": get_feature_set("expanded_vector"),
  "spatial_idw2_all_group": get_feature_set("spatial_idw2_all_group"),
  "spatial_nearest_own_group": get_feature_set("spatial_nearest_own_group"),
}

# 실제값은 원본을 그대로 유지한다. clip은 예측에만 적용한다.
holdoutActual = trainLabels.loc[holdoutMask, TARGET_COLS].copy(deep=True)
actualFingerprint = hashlib.sha256(
  pd.util.hash_pandas_object(holdoutActual, index=True).to_numpy().tobytes()
).hexdigest()


def scoreFeatureSet(config):
  """노트북 08과 동일한 조건으로 2024 holdout 점수를 계산한다."""
  from sklearn.ensemble import RandomForestRegressor
  from sklearn.impute import SimpleImputer

  pipeline = build_feature_pipeline(
    config,
    train_time_index=trainLabels.loc[modelTrainMask, "kst_dtm"],
    test_time_index=trainLabels.loc[holdoutMask, "kst_dtm"],
    train_ldaps=splitFrames["train_ldaps"],
    train_gfs=splitFrames["train_gfs"],
    test_ldaps=splitFrames["holdout_ldaps"],
    test_gfs=splitFrames["holdout_gfs"],
    turbine_locations=turbines if config.uses_spatial else None,
  )
  imputer = SimpleImputer(strategy="median")
  trainImputed = pd.DataFrame(
    imputer.fit_transform(pipeline.train_matrix),
    columns=pipeline.feature_columns,
    index=pipeline.train_matrix.index,
  )
  holdoutImputed = pd.DataFrame(
    imputer.transform(pipeline.test_matrix),
    columns=pipeline.feature_columns,
    index=pipeline.test_matrix.index,
  )

  predictions = pd.DataFrame(index=holdoutActual.index)
  trainSubset = trainLabels.loc[modelTrainMask]
  for target in TARGET_COLS:
    targetColumns = list(pipeline.target_feature_columns[target])
    labels = trainSubset[target]
    nonNull = labels.notna().to_numpy()
    model = RandomForestRegressor(**dict(OFFICIAL_RF_PARAMS))
    model.fit(trainImputed.loc[nonNull, targetColumns], labels.to_numpy()[nonNull])
    predictions[target] = np.clip(
      model.predict(holdoutImputed[targetColumns]), 0, CAPACITY_KWH[target]
    )

  total, oneMinusNmae, ficr = metric(holdoutActual, predictions)
  return pipeline, predictions, total, oneMinusNmae, ficr


scoreRows = []
holdoutPredictions = {}
for name, config in scoreConfigs.items():
  started = time.perf_counter()
  pipeline, predictions, total, oneMinusNmae, ficr = scoreFeatureSet(config)
  holdoutPredictions[name] = predictions
  scoreRows.append(
    {
      "feature_set": name,
      "features": len(pipeline.feature_columns),
      "target_features": min(len(c) for c in pipeline.target_feature_columns.values()),
      "total_score": total,
      "one_minus_nmae": oneMinusNmae,
      "ficr": ficr,
      "seconds": round(time.perf_counter() - started, 1),
    }
  )
  print(f"{name:26s} features={len(pipeline.feature_columns):3d} total={total:.10f}")

scoreFrame = pd.DataFrame(scoreRows).set_index("feature_set")
print(f"\nholdout 실제값 SHA-256 (실행 전후 불변): {actualFingerprint[:16]}...")
scoreFrame

lab_official_mean          features= 74 total=0.5769377449


official_mean              features= 76 total=0.5777425681


expanded_vector            features=319 total=0.5864824799


spatial_idw2_all_group     features=550 total=0.5924854297


spatial_nearest_own_group  features=550 total=0.5898283700

holdout 실제값 SHA-256 (실행 전후 불변): b26ba28bb53d47c6...


,features,target_features,total_score,one_minus_nmae,ficr,seconds
feature_set,,,,,,
lab_official_mean,74,74,0.576938,0.863228,0.290647,2.800000
official_mean,76,76,0.577743,0.863317,0.292168,2.900000
expanded_vector,319,319,0.586482,0.867895,0.305070,5.600000
spatial_idw2_all_group,550,550,0.592485,0.869565,0.315406,14.300000
spatial_nearest_own_group,550,396,0.589828,0.868767,0.310890,13.300000


### 3-1. 랩에 대응하는 네 config가 feature 수와 점수를 그대로 낸다

`lab_official_mean` 74 / `expanded_vector` 319 / `spatial_idw2_all_group` 550 /
`spatial_nearest_own_group` 396(superset 550)으로 랩 기록과 정확히 같다.
`one_minus_nmae`와 `ficr` 분해도 노트북 08의 표와 일치한다.

다섯 번째인 `official_mean`(76)은 랩에 대응 항목이 없다. lead feature 2개가 더 있는
production 기본값이며, G3 판정 대상이 아니라 **S1 제출물의 local 기준선**으로 잰 것이다.

이제 Decision Box ⑪에서 정한 기준으로 G3를 판정한다.

In [6]:
G3_TOLERANCE = 1e-9

gateRows = []
for name in scoreConfigs:
  reproduced = float(scoreFrame.loc[name, "total_score"])
  reported = LAB_REPORTED.get(name)
  fullPrecision = LAB_FULL_PRECISION.get(name)
  if reported is None:
    # official_mean 프리셋은 랩에 대응 항목이 없다. 기준선으로만 쓴다.
    gateRows.append(
      {
        "feature_set": name,
        "재현 점수": reproduced,
        "기준값": np.nan,
        "기준 종류": "랩 대응 없음 — G3 대상 아님",
        "절대 차이": np.nan,
        "판정": "참고",
      }
    )
    continue
  if fullPrecision is None:
    gateRows.append(
      {
        "feature_set": name,
        "재현 점수": reproduced,
        "기준값": reported,
        "기준 종류": "6자리 표기만 존재",
        "절대 차이": abs(reproduced - reported),
        "판정": "6자리 일치" if round(reproduced, 6) == reported else "불일치",
      }
    )
    continue
  difference = abs(reproduced - fullPrecision)
  gateRows.append(
    {
      "feature_set": name,
      "재현 점수": reproduced,
      "기준값": fullPrecision,
      "기준 종류": "노트북 08 전체 정밀도",
      "절대 차이": difference,
      "판정": "통과" if difference <= G3_TOLERANCE else "실패",
    }
  )

gateFrame = pd.DataFrame(gateRows).set_index("feature_set")
fullPrecisionRows = gateFrame[gateFrame["기준 종류"] == "노트북 08 전체 정밀도"]
judgedRows = gateFrame[gateFrame["판정"] != "참고"]
g3Passed = bool((fullPrecisionRows["절대 차이"] <= G3_TOLERANCE).all())
primaryDifference = float(gateFrame.loc["spatial_idw2_all_group", "절대 차이"])

print(f"G3 허용 오차            : {G3_TOLERANCE:.0e}")
print(f"채택 후보 절대 차이     : {primaryDifference:.3e}")
print(f"전체 정밀도 기준 통과   : {g3Passed}  (대상 {len(fullPrecisionRows)}종)")
print(f"6자리 대조 전부 일치    : {bool(judgedRows['판정'].isin(['통과', '6자리 일치']).all())}  (대상 {len(judgedRows)}종)")
with pd.option_context("display.float_format", lambda value: f"{value:.16g}"):
  display(gateFrame)

G3 허용 오차            : 1e-09
채택 후보 절대 차이     : 0.000e+00
전체 정밀도 기준 통과   : True  (대상 3종)
6자리 대조 전부 일치    : True  (대상 4종)


,재현 점수,기준값,기준 종류,절대 차이,판정
feature_set,,,,,
lab_official_mean,0.5769377448952331,0.576938,6자리 표기만 존재,2.551047668664097e-07,6자리 일치
official_mean,0.5777425681025163,NaN,랩 대응 없음 — G3 대상 아님,NaN,참고
expanded_vector,0.5864824798698839,0.5864824798698839,노트북 08 전체 정밀도,0,통과
spatial_idw2_all_group,0.5924854297224146,0.5924854297224146,노트북 08 전체 정밀도,0,통과
spatial_nearest_own_group,0.5898283700164271,0.5898283700164271,노트북 08 전체 정밀도,0,통과


### 3-2. G3 통과 — 채택 후보가 부동소수점 단위로 랩과 같다

`spatial_idw2_all_group`의 재현 점수가 노트북 08의 `0.5924854297224146`과
`1e-9` 이내로 일치한다. `expanded_vector`와 `spatial_nearest_own_group`도 역산한
전체 정밀도 기준을 통과했다. `lab_official_mean`은 08이 전체 정밀도를 남기지 않아
6자리 대조만 가능했고, 그 범위에서 일치한다.

이것이 뜻하는 바는 단순하지 않다. 점수가 같다는 것은 **550개 컬럼의 이름과 순서,
grid 통계의 자유도(`std_ddof=0`), pooling 가중치, imputer를 fit한 17,519행,
target별 non-null mask, 예측에만 적용한 clip이 모두 노트북 08과 동일**하다는 뜻이다.
어느 하나라도 달랐다면 `max_features="sqrt"`인 RandomForest는 다른 트리를 만들었을 것이다.

**G3 게이트를 통과했으므로 제출 단계로 진행한다.**

---

## 4. 사다리를 다시 읽는다 — 개선분은 어디서 왔는가

점수가 재현됐으니 이제 그 점수가 무엇을 말하는지 읽을 수 있다.

In [7]:
PREVIOUS_STEP = {
  "lab_official_mean": None,
  "official_mean": "lab_official_mean",
  "expanded_vector": "official_mean",
  "spatial_idw2_all_group": "expanded_vector",
  "spatial_nearest_own_group": "expanded_vector",
}

ladder = scoreFrame.copy()
ladder["Δ vs 통제군"] = ladder["total_score"] - ladder.loc["lab_official_mean", "total_score"]
ladder["Δ vs 직전 단계"] = [
  np.nan
  if PREVIOUS_STEP[name] is None
  else ladder.loc[name, "total_score"] - ladder.loc[PREVIOUS_STEP[name], "total_score"]
  for name in ladder.index
]
ladder["Δ 1-NMAE"] = ladder["one_minus_nmae"] - ladder.loc["lab_official_mean", "one_minus_nmae"]
ladder["Δ FICR"] = ladder["ficr"] - ladder.loc["lab_official_mean", "ficr"]
ladder["FICR 기여율"] = (0.5 * ladder["Δ FICR"]) / ladder["Δ vs 통제군"].replace(0, np.nan)

display(ladder[["features", "total_score", "Δ vs 통제군", "Δ vs 직전 단계"]])
display(ladder[["Δ 1-NMAE", "Δ FICR", "FICR 기여율"]])

,features,total_score,Δ vs 통제군,Δ vs 직전 단계
feature_set,,,,
lab_official_mean,74,0.576938,0.000000,NaN
official_mean,76,0.577743,0.000805,0.000805
expanded_vector,319,0.586482,0.009545,0.008740
spatial_idw2_all_group,550,0.592485,0.015548,0.006003
spatial_nearest_own_group,550,0.589828,0.012891,0.003346


,Δ 1-NMAE,Δ FICR,FICR 기여율
feature_set,,,
lab_official_mean,0.000000,0.000000,NaN
official_mean,0.000089,0.001521,0.944759
expanded_vector,0.004667,0.014423,0.755537
spatial_idw2_all_group,0.006337,0.024759,0.796215
spatial_nearest_own_group,0.005539,0.020243,0.785169


### 4-1. 개선분의 80%가 FICR에서 나왔다

`spatial_idw2_all_group`의 총 개선 +0.015548을 분해하면 1-NMAE 기여가 +0.003169,
FICR 기여가 +0.012380이다(각각 0.5 가중). FICR 기여율 **79.6%** — 즉 평균 오차가 줄어든
것보다 **정산 구간에 들어간 시각이 늘어난 효과가 네 배 가까이 크다.**

표의 두 번째 행도 함께 읽을 만하다. `official_mean`(76)은 랩 통제군(74)에 lead feature
2개를 더한 것뿐인데 +0.000805가 나왔다. 노트북 09의 Decision Box ①은 "기본값을 랩
통제군이 아니라 production 경로에 맞춘다"고 결정했는데, 그 선택이 점수 면에서도
손해가 아니었음이 여기서 확인된다.

공식 산식이 오차율 6% 이하에 단가 4.0, 8% 이하에 3.0, 그 위에 0을 주는 계단 함수라
이런 비대칭이 생긴다. 평균을 조금 줄이는 것보다 **경계 근처의 시각을 6% 안으로
밀어 넣는 것**이 점수에 훨씬 크게 반영된다.

이 관찰은 다음 작업(metric-aware calibration)의 근거가 되지만, 동시에 경고이기도 하다.
FICR은 계단 함수라 임계값 근처에서 불안정하다. holdout에서 6% 경계를 아슬아슬하게
넘긴 시각이 Public에서는 넘지 못할 수 있고, 그 경우 개선분이 local보다 작게 나타난다.

---

## 5. 제출 후보 S1·S2 만들기

G3를 통과했으므로 제출물을 만든다. 여기서부터는 holdout 실험이 아니라 **실제 제출 경로**다.

### Decision Box ⑬ — 제출 모델은 holdout 실험 모델을 재사용하지 않고 2022~2024 전체로 다시 학습한다

**선택지**

- (A) 3절에서 만든 2022~2023 학습 모델을 그대로 써서 2025를 예측한다
- (B) 라벨 전체(2022~2025-01-01 00:00, 26,304행)로 다시 학습한다

**근거**

(A)는 holdout 점수와 제출물이 같은 모델에서 나오므로 대응이 명확하다는 장점이 있다.
그러나 2024년 한 해의 라벨 8,784행을 버리는 것이며, 예측 대상인 2025년과 가장 가까운
기간을 빼고 학습하는 셈이다. 시계열에서 최근 데이터를 버리는 비용은 크다.

(B)는 학습 데이터가 50% 늘어난다. 2절에서 확인했듯 라벨의 마지막 시각이 제출 시간축의
첫 시각보다 앞서므로 look-ahead도 없다. 대신 **제출물의 점수를 예측할 근거가
holdout 점수뿐**이라는 한계가 생긴다 — 이 모델은 검증된 적이 없다.

**채택: (B)** — 단, ledger에 "holdout 점수는 2022~2023 학습 모델의 것이고 제출물은
전체 학습 모델"이라는 사실을 명시해 둔다. 두 숫자를 같은 것으로 읽으면 안 된다.

제출은 production CLI 경로로 만든다. 노트북에서 파이프라인을 직접 부르면
`train.py`가 실제로 동작하는지는 검증되지 않기 때문이다.

In [8]:
from baram.inference import main as inferenceMain
from baram.train import main as trainMain

modelDir = outputRoot / "models"
submissionDir = outputRoot / "submissions"
modelDir.mkdir(parents=True, exist_ok=True)
submissionDir.mkdir(parents=True, exist_ok=True)
runRegistry = outputRoot / "run_registry.csv"

SUBMISSION_SLOTS = {
  "S1": {
    "feature_set": "official_mean",
    "purpose": "형식·인코딩·행수·파이프라인 smoke",
    "experiment_id": "s1_official_mean_smoke",
  },
  "S2": {
    "feature_set": "spatial_idw2_all_group",
    "purpose": "채택 후보 — local Δ가 Public에서도 같은 방향인지",
    "experiment_id": "s2_spatial_idw2_all_group",
  },
}


def trainSubmissionModel(slot):
  """production train CLI로 전체 라벨을 학습한다."""
  spec = SUBMISSION_SLOTS[slot]
  featureSet = spec["feature_set"]
  modelPath = modelDir / f"{slot.lower()}_{featureSet}.pkl"
  argv = [
    "--run",
    "--train-labels", str(dataDir / "train" / "train_labels.csv"),
    "--ldaps-train", str(dataDir / "train" / "ldaps_train.csv"),
    "--gfs-train", str(dataDir / "train" / "gfs_train.csv"),
    "--model-output", str(modelPath),
    "--registry-output", str(runRegistry),
    "--data-manifest", str(dataDir / "MANIFEST.md"),
    "--experiment-id", spec["experiment_id"],
    "--feature-set", featureSet,
    "--split-name", "holdout_2024",
    "--train-start-kst", str(holdoutSplitBounds["train_start"]),
    "--train-end-kst", str(holdoutSplitBounds["train_end"]),
    "--validation-start-kst", str(holdoutSplitBounds["holdout_start"]),
    "--validation-end-kst", str(holdoutSplitBounds["holdout_end"]),
  ]
  # 같은 피처셋의 holdout 성적을 기록한다. 제출 모델 자체의 점수가 아니다.
  if featureSet in scoreFrame.index:
    argv += [
      "--total-score", f"{scoreFrame.loc[featureSet, 'total_score']:.10f}",
      "--one-minus-nmae", f"{scoreFrame.loc[featureSet, 'one_minus_nmae']:.10f}",
      "--ficr", f"{scoreFrame.loc[featureSet, 'ficr']:.10f}",
    ]
  if get_feature_set(featureSet).uses_spatial:
    argv += ["--info-xlsx", str(dataDir / "info.xlsx")]
  started = time.perf_counter()
  trainMain(argv)
  return modelPath, round(time.perf_counter() - started, 1)


s1ModelPath, s1TrainSeconds = trainSubmissionModel("S1")
print(f"\nS1 학습 완료: {s1TrainSeconds}초")

saved_model=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\models\s1_official_mean.pkl
saved_metadata=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\models\s1_official_mean.pkl.metadata.json
run_registry=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\run_registry.csv
run_id=20260727T103911-s1_official_mean_smoke-2af32c98-d307655

S1 학습 완료: 6.3초


S1 모델이 저장되고 run registry에 행이 하나 쌓였다. `run_id`와 `preprocessing_sha256`가
출력에 남는다. 여기서 쓴 `--split-name`·`--total-score` 인자는 3절 holdout 실험의
결과이며, Decision Box ⑬에 따라 제출 모델 자체의 점수가 아니라 **같은 피처셋의
holdout 성적**이라는 뜻이다.

이제 inference CLI로 제출 CSV를 만든다. 피처셋은 인자로 주지 않는다 —
metadata sidecar에서 자동 복원되는지가 설계서 6.2절의 계약이다.

In [9]:
def buildSubmission(slot, modelPath):
  """production inference CLI로 제출 CSV를 만든다. 피처셋은 metadata에서 복원된다."""
  submissionPath = submissionDir / f"{slot.lower()}_{SUBMISSION_SLOTS[slot]['feature_set']}.csv"
  argv = [
    "--run",
    "--model-input", str(modelPath),
    "--run-registry", str(runRegistry),
    "--sample-submission", str(dataDir / "sample_submission.csv"),
    "--ldaps-test", str(dataDir / "test" / "ldaps_test.csv"),
    "--gfs-test", str(dataDir / "test" / "gfs_test.csv"),
    "--submission-output", str(submissionPath),
  ]
  started = time.perf_counter()
  inferenceMain(argv)
  return submissionPath, round(time.perf_counter() - started, 1)


from baram.registry import default_metadata_path

s1SubmissionPath, s1InferSeconds = buildSubmission("S1", s1ModelPath)
s1Metadata = json.loads(default_metadata_path(s1ModelPath).read_text(encoding="utf-8"))
print(f"\nS1 추론 완료: {s1InferSeconds}초")
print("metadata에 각인된 피처셋:", json.dumps(s1Metadata["feature_set"], ensure_ascii=False)[:200])

saved_submission=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\submissions\s1_official_mean.csv
submission_sha256=5f29ca511e44d11664b370adbdc71afbf9b367927a31e557a1be6e32cacbe420

S1 추론 완료: 1.3초
metadata에 각인된 피처셋: {"config": {"calendar": true, "include_lead": true, "name": "official_mean", "spatial": null, "statistics": ["mean"], "wind_vector": false}, "feature_count": 76, "name": "official_mean", "resolved_sha


### 5-1. 피처셋을 인자로 주지 않아도 metadata에서 복원된다

`--feature-set`을 넘기지 않았는데도 inference가 올바른 피처셋으로 550개든 76개든
같은 컬럼을 만들어 냈다. metadata sidecar의 `feature_set` 블록에 이름·scope·
feature 수·resolved hash가 각인되어 있고, inference는 그것으로 config를 되살린다.

이 계약이 없다면 train은 `spatial_idw2_all_group`으로 학습하고 inference는 기본값
`official_mean`으로 피처를 만드는 사고가 조용히 일어날 수 있다. 컬럼 수가 다르므로
어딘가에서 실패하기는 하겠지만, 실패 지점이 "왜 실패했는지 알기 어려운 곳"이 된다.

이제 S2 채택 후보를 같은 경로로 만든다.

In [10]:
s2ModelPath, s2TrainSeconds = trainSubmissionModel("S2")
s2SubmissionPath, s2InferSeconds = buildSubmission("S2", s2ModelPath)
s2Metadata = json.loads(default_metadata_path(s2ModelPath).read_text(encoding="utf-8"))

print(f"\nS2 학습 {s2TrainSeconds}초 / 추론 {s2InferSeconds}초")
print("S2 metadata 피처셋:", json.dumps(s2Metadata["feature_set"], ensure_ascii=False)[:220])
print("schema version:", s1Metadata.get("schema_version"), "/", s2Metadata.get("schema_version"))

saved_model=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\models\s2_spatial_idw2_all_group.pkl
saved_metadata=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\models\s2_spatial_idw2_all_group.pkl.metadata.json
run_registry=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\run_registry.csv
run_id=20260727T103935-s2_spatial_idw2_all_group-34c2b1a8-d307655


saved_submission=C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\.claude\worktrees\codex-to-claude-transition-a9327d\outputs\submissions\s2_spatial_idw2_all_group.csv
submission_sha256=853d3218c1e49e7757d2e1fbe9d1a6fccbab76d384a3a04b176634e764b9c747

S2 학습 21.9초 / 추론 3.6초
S2 metadata 피처셋: {"config": {"calendar": true, "include_lead": true, "name": "spatial_idw2_all_group", "spatial": {"idw_power": "2", "methods": ["idw"], "scope": "all_group"}, "statistics": ["mean", "std", "min", "max"], "wind_vector": t
schema version: 1.1 / 1.1


두 슬롯의 모델과 제출 CSV가 모두 만들어졌다. metadata schema는 1.1이며
`feature_set` 블록이 서로 다른 hash를 기록한다.

이제 제출물 자체를 검증한다. 형식이 아니라 **내용**을 본다.

In [11]:
from baram.validation import build_validated_submission_artifact


def describeSubmission(slot, submissionPath):
  # validator는 sample_submission의 원본 문자열과 대조하므로 dtype을 바꾸지 않는다.
  frame = pd.read_csv(submissionPath, encoding="utf-8-sig")
  artifact = build_validated_submission_artifact(
    frame,
    sampleSubmission,
    encoding="utf-8-sig",
  )
  digest = hashlib.sha256(submissionPath.read_bytes()).hexdigest()
  record = {
    "slot": slot,
    "feature_set": SUBMISSION_SLOTS[slot]["feature_set"],
    "rows": len(frame),
    "validator": "통과",
    "sha256": digest[:16] + "...",
    "파일": submissionPath.name,
  }
  for target in TARGET_COLS:
    record[f"{target} 평균"] = float(frame[target].mean())
  return record, frame, digest


s1Record, s1Frame, s1Digest = describeSubmission("S1", s1SubmissionPath)
s2Record, s2Frame, s2Digest = describeSubmission("S2", s2SubmissionPath)

submissionFrame = pd.DataFrame([s1Record, s2Record]).set_index("slot")
identicalFiles = s1Digest == s2Digest
predictionGap = pd.DataFrame(
  {
    "S2 - S1 평균": [float((s2Frame[t] - s1Frame[t]).mean()) for t in TARGET_COLS],
    "S2 - S1 절대차 평균": [float((s2Frame[t] - s1Frame[t]).abs().mean()) for t in TARGET_COLS],
    "상관계수": [float(s1Frame[t].corr(s2Frame[t])) for t in TARGET_COLS],
  },
  index=TARGET_COLS,
)
print(f"두 제출물이 같은 파일인가 : {identicalFiles}  (달라야 정상)")
print(f"행 수 대조 : sample_submission {len(sampleSubmission)} / S1 {len(s1Frame)} / S2 {len(s2Frame)}")
display(submissionFrame)
display(predictionGap)

두 제출물이 같은 파일인가 : False  (달라야 정상)
행 수 대조 : sample_submission 8760 / S1 8760 / S2 8760


,feature_set,rows,validator,sha256,파일,kpx_group_1 평균,kpx_group_2 평균,kpx_group_3 평균
slot,,,,,,,,
S1,official_mean,8760,통과,5f29ca511e44d116...,s1_official_mean.csv,7359.033109,7875.371758,6335.105336
S2,spatial_idw2_all_group,8760,통과,853d3218c1e49e77...,s2_spatial_idw2_all_group.csv,7382.550549,7891.992502,6323.576805


,S2 - S1 평균,S2 - S1 절대차 평균,상관계수
kpx_group_1,23.517440,522.349617,0.991557
kpx_group_2,16.620744,590.375608,0.990743
kpx_group_3,-11.528531,510.430495,0.990917


### 5-2. 두 제출물은 형식은 같고 내용은 다르다

행 수 8,760으로 sample_submission과 일치하고, validator가 두 파일 모두 통과했다.
SHA-256은 서로 다르다 — 같았다면 피처셋 인자가 실제로는 반영되지 않았다는 뜻이므로
그 자체가 사고다.

예측값은 target별 상관계수가 0.99로 높지만 절대차 평균이 510~590 kWh다.
설비용량 21,600 kWh 대비 약 2.4~2.7%이며, 공식 산식의 정산 경계가 오차율 6%인 것을
생각하면 결코 작지 않은 값이다. 두 모델이 대체로 같은 방향을 보되 **정산 구간 경계를
서로 다르게 넘나든다**는 뜻이다.

여기서 비교 기준을 정확히 해 둔다. 이번 제출로 확인하려는 local Δ는 랩 통제군 대비
+0.015548이 아니라 **S1↔S2의 +0.014742**(0.592485 − 0.577743)다. S1이 쓰는 것은 74개
랩 통제군이 아니라 76개 production 기본값이기 때문이다.

### Decision Box ⑭ — S1은 버리는 슬롯이 아니라 기울기 측정의 기준점이다

**선택지**

- (A) 채택 후보 S2 하나만 제출해 슬롯을 아낀다
- (B) S1(통제군)과 S2(채택 후보)를 같은 날 제출한다

**근거**

리더보드 제출 이력이 **0회**인 상태에서 S2만 올리면 점수 하나를 얻는다. 그런데 그 점수가
좋은지 나쁜지 판단할 기준이 없다. local 0.5925가 Public에서 0.55로 나왔다면 그것이
피처셋 문제인지, 2025년 분포 변화인지, 제출 형식 문제인지 구분할 수 없다.

S1을 함께 올리면 **두 점**이 생긴다. 이때 비교할 local Δ는 **+0.014742**(S2 0.592485 −
S1 0.577743)이며, 설계서 8절이 적은 +0.015547은 랩 통제군(74개) 기준이라 이번 두 슬롯의
대조에는 맞지 않는다. 이 값과 Public Δ를 비교하면 "local이 이만큼 오르면 Public은 대략
이만큼"이라는 환산이 가능해진다. 하루 5회 한도에서 2회를 쓰는 비용은 이 정보에 비해 싸다.

**채택: (B)** — S3~S5는 이번 작업에서 소모하지 않는다.

### Decision Box ⑮ — 같은 holdout을 계속 재사용하는 비용을 기록으로 남긴다

**선택지**

- (A) 2024 holdout 점수를 계속 최적화 기준으로 쓴다
- (B) holdout 재사용 횟수를 세고, Public 신호가 들어오면 판단 기준을 옮긴다

**근거**

노트북 06·07·08과 이 노트북까지, 2024 holdout은 이미 여러 차례 피처 선택 근거로
쓰였다. Dwork 등이 정리한 대로 같은 검증셋을 적응적으로 반복 사용하면 그 추정치는
점점 낙관적으로 편향되며(*The reusable holdout*, Science 349, 2015), Blum과 Hardt는
리더보드 자체가 같은 문제를 겪는다는 것을 보였다(*The Ladder*, ICML 2015).

우리 상황에서 이 위험은 아직 크지 않다 — 비교한 후보가 8개 남짓이고, 개선분이
+0.0156으로 노이즈 수준을 크게 넘는다. 그러나 앞으로 GBM·fold·calibration으로
후보가 수십 개가 되면 이야기가 달라진다.

**채택: (B)** — 이번 제출로 얻는 Public 2점을 **holdout 추정치의 편향을 재는 자**로 쓴다.
local Δ와 Public Δ의 비율이 1보다 한참 작으면 그것이 곧 holdout 과적합의 증거다.

아래에서 제출 ledger 행을 준비한다. 제출 후 Public 점수를 채워 넣을 자리를 함께 만든다.

In [12]:
import subprocess

commitSha = subprocess.run(
  ["git", "rev-parse", "HEAD"],
  cwd=projectRoot,
  capture_output=True,
  text=True,
  check=False,
).stdout.strip()[:12]

registryFrame = pd.read_csv(runRegistry, encoding="utf-8-sig")
latestRuns = registryFrame.tail(2)

ledgerRows = []
for slot, submissionPath, digest, metadata in [
  ("S1", s1SubmissionPath, s1Digest, s1Metadata),
  ("S2", s2SubmissionPath, s2Digest, s2Metadata),
]:
  spec = SUBMISSION_SLOTS[slot]
  scoreKey = spec["feature_set"]
  ledgerRows.append(
    {
      "slot": slot,
      "목적": spec["purpose"],
      "experiment_id": spec["experiment_id"],
      "feature_set": spec["feature_set"],
      "run_id": metadata["run_id"],
      "commit": commitSha,
      "seed": OFFICIAL_RF_PARAMS["random_state"],
      "학습 범위": "2022-01-01 01:00 ~ 2025-01-01 00:00 (26,304행)",
      "local holdout total": round(float(scoreFrame.loc[scoreKey, "total_score"]), 6),
      "local 기준": "2024 holdout · 2022~2023 학습 모델",
      "제출 파일": submissionPath.name,
      "submission_sha256": digest,
      "preprocessing_sha256": metadata["preprocessing_sha256"][:16] + "...",
      "public_score": None,
      "dacon_submission_id": None,
      "예상 리스크": (
        "2025 LDAPS 결측 133컬럼을 train median으로 대체 — local에 없는 조건"
        if slot == "S2"
        else "승격 전 baseline과 동일 동작 확인용"
      ),
    }
  )

ledger = pd.DataFrame(ledgerRows).set_index("slot")
ledgerPath = outputRoot / "submission_ledger_draft.csv"
ledger.to_csv(ledgerPath, encoding="utf-8-sig")

print("run registry 최근 2행의 run_id:", list(latestRuns["run_id"]))
print(f"ledger 초안 저장: {ledgerPath.name}")
display(ledger.T)

run registry 최근 2행의 run_id: ['20260727T103911-s1_official_mean_smoke-2af32c98-d307655', '20260727T103935-s2_spatial_idw2_all_group-34c2b1a8-d307655']
ledger 초안 저장: submission_ledger_draft.csv


slot,S1,S2
목적,형식·인코딩·행수·파이프라인 smoke,채택 후보 — local Δ가 Public에서도 같은 방향인지
experiment_id,s1_official_mean_smoke,s2_spatial_idw2_all_group
feature_set,official_mean,spatial_idw2_all_group
run_id,20260727T103911-s1_official_mean_smoke-2af32c9...,20260727T103935-s2_spatial_idw2_all_group-34c2...
commit,d307655fb473,d307655fb473
seed,42,42
학습 범위,"2022-01-01 01:00 ~ 2025-01-01 00:00 (26,304행)","2022-01-01 01:00 ~ 2025-01-01 00:00 (26,304행)"
local holdout total,0.577743,0.592485
local 기준,2024 holdout · 2022~2023 학습 모델,2024 holdout · 2022~2023 학습 모델
제출 파일,s1_official_mean.csv,s2_spatial_idw2_all_group.csv


### 5-3. ledger는 제출 전 절반, 제출 후 절반이 채워진다

`public_score`와 `dacon_submission_id`가 비어 있다. 이 두 칸은 사람이 제출 버튼을 누른 뒤
채워지며, 그때 비로소 local↔Public 기울기를 계산할 수 있다.

`local holdout total`과 제출물의 관계를 다시 확인해 둔다. 이 점수는 2022~2023으로 학습한
모델이 2024에서 받은 성적이고, 제출물은 2022~2024 전체로 학습한 **다른 모델**이 만든 것이다.
Decision Box ⑬의 대가이며, ledger의 `local 기준` 칸이 그 사실을 기록한다.

---

## 6. 종합 결론

### 6-1. 연구 질문

> 승격한 production 파이프라인이 랩에서 잰 점수를 그대로 재현하고, 제출물을 만들 수 있는가?

**재현하고, 만들 수 있다.** `spatial_idw2_all_group`의 2024 holdout total score가
노트북 08의 전체 정밀도 값과 `1e-9` 이내로 일치했고(G3 통과), 같은 파이프라인이
production CLI를 통해 validator를 통과하는 제출 CSV 두 벌을 만들어 냈다.

### 6-2. 단계별 요약

| 절 | 확인한 것 | 결과 |
|----|-----------|------|
| 1 | 재현 판정 기준 | 6자리 표기가 아니라 전체 정밀도 값 기준 (Decision Box ⑪) |
| 2 | holdout 분할 | 17,519 / 8,784 / 경계 행 1건 제외 — 노트북 08과 동일 |
| 3 | 점수 체인 | 프리셋 4종 feature 수·점수 모두 재현, **G3 통과** |
| 4 | 개선분 분해 | 총 +0.015548 중 FICR 기여율 79.6% |
| 5 | 제출 후보 | S1·S2 validator 통과, 서로 다른 SHA-256, ledger 초안 생성 |

### 6-3. 주요 발견

1. **재현 판정은 기준값의 정밀도부터 정해야 한다.** 문서에 적힌 `0.592485`를 그대로
   기준 삼았다면 표기 오차 5e-7 때문에 정상 재현이 `1e-9` 게이트에서 실패로 나왔을 것이다.
   재현성 게이트는 허용 오차만이 아니라 **비교 대상**까지 명세해야 한다.
2. **개선분의 79.6%가 FICR에서 왔다.** 공식 산식이 계단 함수라 평균 오차 감소보다
   정산 구간 진입이 점수에 크게 반영된다. 동시에 이는 임계값 근처의 불안정성을 뜻하므로,
   local Δ가 Public에서 그대로 재현되지 않을 수 있는 첫 번째 이유다. S1과 S2의 예측이
   상관계수 0.99에 절대차 평균 520 kWh인 것도 같은 이야기다 — 두 모델의 차이는 크기가
   아니라 **경계를 넘나드는 방식**에 있다.
3. **local과 Public이 어긋날 구조적 이유가 이미 있다.** 10번 노트북에서 확인했듯
   2025년 test는 133개 컬럼에 결측이 있고 train median으로 채워지는데, 2024 holdout에는
   그런 결측이 없다. 즉 제출물은 local이 한 번도 겪지 않은 조건에서 예측한다.

### 6-4. 시사점

승격 작업의 가치는 점수 +0.0156이 아니라 **그 점수가 어디서 왔는지 추적 가능해졌다**는
데 있다. 제출물마다 `preprocessing_sha256`와 `feature_set` 블록이 각인되고, run registry가
run_id와 submission hash를 잇는다. Public 점수가 돌아왔을 때 "어떤 전처리의 결과인가"를
파일 하나로 답할 수 있으며, 이것이 없으면 두 점을 얻어도 기울기를 계산할 대상이 불분명하다.

### 6-5. 한계

- **제출 모델은 검증된 적이 없다.** holdout 점수는 2022~2023 학습 모델의 것이고,
  제출물은 2022~2024 전체 학습 모델이 만들었다(Decision Box ⑬). 두 숫자를 같게 읽으면 안 된다.
- `lab_official_mean`은 노트북 08이 전체 정밀도를 남기지 않아 **6자리 대조만** 가능했다.
  나머지 세 프리셋과 달리 `1e-9` 수준의 보증이 없다.
- 2024 holdout은 이미 여러 차례 피처 선택에 재사용됐다(Decision Box ⑮). 현재 개선분이
  노이즈를 크게 넘어 위험이 작지만, 후보가 늘어나면 추정치의 낙관 편향이 커진다.
- 이번 작업은 **모델을 하나도 바꾸지 않았다.** RandomForest 공식 파라미터 그대로이며,
  GBM·fold·calibration은 설계서 3.2절에 따라 범위 밖이다.

### 6-6. 요약

승격한 파이프라인은 노트북 08의 `0.5924854297224146`을 부동소수점 단위로 재현했고,
production CLI를 통해 validator를 통과한 제출 후보 두 벌과 ledger 초안을 만들었다.
S1은 승격 전 동작을 확인하는 기준점이고 S2는 채택 후보이며, 두 점의 Public 점수가
들어오면 local Δ +0.014742에 대응하는 Public Δ로 처음 기울기를 추정할 수 있다.

**제출 버튼은 사람이 직접 누른다.** 이 노트북의 역할은 여기까지다.

---

## 산출물 확인

이 노트북이 만든 파일을 명시적으로 나열한다.

In [13]:
createdArtifacts = sorted(
  str(path.relative_to(projectRoot)).replace("\\", "/")
  for path in outputRoot.rglob("*")
  if path.is_file()
)
print("생성한 산출물:")
for name in createdArtifacts:
  print("  -", name)
print()
print("outputs/가 .gitignore 대상인가 :", "outputs/**" in (projectRoot / ".gitignore").read_text(encoding="utf-8"))
print("data/raw/open 원본 수정 여부   :", ldapsTrain.shape == (420864, 35))
print("holdout 실제값 SHA-256 불변    :", hashlib.sha256(
  pd.util.hash_pandas_object(trainLabels.loc[holdoutMask, TARGET_COLS], index=True).to_numpy().tobytes()
).hexdigest() == actualFingerprint)

생성한 산출물:
  - outputs/models/s1_official_mean.pkl
  - outputs/models/s1_official_mean.pkl.metadata.json
  - outputs/models/s2_spatial_idw2_all_group.pkl
  - outputs/models/s2_spatial_idw2_all_group.pkl.metadata.json
  - outputs/run_registry.csv
  - outputs/submission_ledger_draft.csv
  - outputs/submissions/s1_official_mean.csv
  - outputs/submissions/s2_spatial_idw2_all_group.csv

outputs/가 .gitignore 대상인가 : True
data/raw/open 원본 수정 여부   : True
holdout 실제값 SHA-256 불변    : True


제출 후보와 모델은 `outputs/` 아래에만 만들어졌고 이 경로는 커밋되지 않는다.
공식 원자료는 읽기만 했으며 holdout 실제값도 실행 전후로 동일하다.

다음 단계는 사람의 제출과 Public 점수 회수이며, 그 결과를 ledger의 빈 두 칸에 채우면
이번 승격 작업이 닫힌다.